# Laboratorio IA — Clasificación Multiclase con Regresión Logística
## Dataset: ExtraSensory — Reconocimiento de Actividades Humanas

**Estudiante:** Jorge Gabriel Escobar Moscoso  
**Docente:** Ariel Guarachi — 63706640

---

### Descripción del Dataset

El **ExtraSensory Dataset** fue recopilado en condiciones reales (*in-the-wild*) usando smartphones y wearables. Contiene lecturas de sensores —acelerómetro, giroscopio, magnetómetro, audio y GPS— de 60 usuarios en su vida cotidiana.

| Característica | Valor |
|---|---|
| Número de features (n) | 225 |
| Número de ejemplos (m) | ≥ 300 000 |
| Tipo | Tabular numérico (no gráfico) |
| Tarea | Clasificación multiclase (actividad humana) |
| Clases | LYING_DOWN, SITTING, STANDING, WALKING, RUNNING, BICYCLING |

Cumple los requisitos: **n ≥ 40** y **m ≥ 50 000**.

> **Fuente:** https://www.kaggle.com/datasets/yvaizman/the-extrasensory-dataset

---

### Estrategia general

Se implementa regresión logística **One-vs-Rest (OvR)**: para cada clase *k* se entrena un clasificador binario independiente que decide "¿es clase k o no?". La predicción final asigna la clase cuyo clasificador entregó la mayor probabilidad.

El dataset se divide en **80% entrenamiento / 20% prueba**, sin solapamiento. Se aplica balanceo de clases con submuestreo y normalización Z-score con Pandas.

### 1. Montar Google Drive y cargar dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
%matplotlib inline

### 2. Carga y exploración del dataset

El dataset se distribuye como un archivo CSV consolidado. Se cargan los datos con Pandas para aprovechar sus herramientas de exploración y preprocesamiento.

Las columnas de etiquetas comienzan con `label:`. Se seleccionan únicamente las 6 actividades principales para la clasificación multiclase.

In [ ]:
# ─── AJUSTAR ESTA RUTA AL ARCHIVO EN TU DRIVE ───────────────────────────────
DATA_PATH = '/content/drive/MyDrive/Datasets/extrasensory_merged.csv'
# ─────────────────────────────────────────────────────────────────────────────

df_raw = pd.read_csv(DATA_PATH)
print(f'Shape original: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Columnas de etiqueta disponibles
label_cols = [c for c in df_raw.columns if c.startswith('label:')]
print(f'Columnas de etiqueta encontradas ({len(label_cols)}):')  
print(label_cols)

### 3. Preprocesamiento con Pandas

#### 3.1 Selección de clases y construcción del vector de etiquetas

Se eligen 6 actividades mutuamente excluyentes. Para cada fila se asigna la etiqueta de la actividad cuya columna binaria vale `1.0`. Filas con ninguna o más de una actividad activa se descartan para mantener la consistencia.

In [ ]:
# Clases objetivo — ajustar si el CSV usa nombres distintos
CLASES = [
    'label:LYING_DOWN',
    'label:SITTING',
    'label:STANDING',
    'label:WALKING',
    'label:RUNNING',
    'label:BICYCLING'
]

# Filtrar solo filas que tienen exactamente una clase activa
mask = df_raw[CLASES].sum(axis=1) == 1
df = df_raw[mask].copy()
print(f'Filas con exactamente una clase activa: {len(df)}')

# Construir vector de etiquetas numérico (0..5)
df['clase'] = df[CLASES].values.argmax(axis=1)
print('Distribución de clases:')
print(df['clase'].value_counts().sort_index())

#### 3.2 Balanceo de clases

Para garantizar que el modelo no esté sesgado hacia las clases mayoritarias, se aplica **submuestreo** (undersampling): se toma la clase con menos ejemplos como referencia y se reduce todas las demás al mismo número.

In [ ]:
min_count = df['clase'].value_counts().min()
print(f'Ejemplos por clase tras balanceo: {min_count}')

grupos = []
for cls in range(len(CLASES)):
    subset = df[df['clase'] == cls].sample(n=min_count, random_state=42)
    grupos.append(subset)

df_bal = pd.concat(grupos).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Shape balanceado: {df_bal.shape}')
print('Distribución tras balanceo:')
print(df_bal['clase'].value_counts().sort_index())

In [ ]:
# Visualizar distribución balanceada
nombre_clases = ['LYING_DOWN','SITTING','STANDING','WALKING','RUNNING','BICYCLING']
counts = df_bal['clase'].value_counts().sort_index()

pyplot.figure(figsize=(9,4))
pyplot.bar(nombre_clases, counts.values, color='steelblue', edgecolor='black')
pyplot.xlabel('Actividad')
pyplot.ylabel('Número de ejemplos')
pyplot.title('Distribución de clases — Dataset balanceado')
pyplot.tight_layout()
pyplot.show()

#### 3.3 Separación de features y etiquetas

Se identifican las columnas de sensores (numéricas, excluyendo columnas de etiqueta y metadatos). Los valores NaN se reemplazan por la media de cada columna —imputación simple— y luego se aplica normalización **Z-score** para que todos los features tengan media 0 y desviación estándar 1.

In [ ]:
# Columnas de features: todas las numéricas excepto etiquetas y metadatos
excluir = CLASES + ['clase', 'timestamp', 'uuid']
feature_cols = [c for c in df_bal.columns if c not in excluir and df_bal[c].dtype in ['float64','float32','int64','int32']]
print(f'Número de features seleccionadas (n): {len(feature_cols)}')

X_df = df_bal[feature_cols].copy()
y_arr = df_bal['clase'].values

# Imputación de NaN con la media de cada columna
X_df = X_df.fillna(X_df.mean())
print(f'NaN restantes: {X_df.isna().sum().sum()}')

In [ ]:
def featureNormalize(X):
    """Normalización Z-score: (X - media) / desv_estándar"""
    mu    = X.mean(axis=0)
    sigma = X.std(axis=0)
    sigma[sigma == 0] = 1  # evitar división por cero
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_np = X_df.values
X_norm, mu, sigma = featureNormalize(X_np)
print(f'Media (primeras 5 cols): {X_norm.mean(axis=0)[:5].round(6)}')
print(f'Desv.std (primeras 5 cols): {X_norm.std(axis=0)[:5].round(6)}')

#### 3.4 División 80/20 — Entrenamiento y Prueba

Se reserva el **20% final** del dataset mezclado para prueba. Este subconjunto **no participa en ninguna etapa de entrenamiento** (ni en el ajuste de parámetros ni en la normalización calculada sobre los datos de entrenamiento).

La normalización se calcula exclusivamente sobre el conjunto de entrenamiento; los parámetros `mu` y `sigma` se reutilizan para transformar el conjunto de prueba.

In [ ]:
m_total = X_norm.shape[0]
m_train = int(0.8 * m_total)

X_train = X_norm[:m_train]
y_train = y_arr[:m_train]
X_test  = X_norm[m_train:]
y_test  = y_arr[m_train:]

print(f'Total ejemplos     : {m_total}')
print(f'Entrenamiento (80%): {X_train.shape[0]}')
print(f'Prueba        (20%): {X_test.shape[0]}')

# Agregar término de intercepción (columna de unos)
m_tr, n_feat = X_train.shape
X_train_b = np.concatenate([np.ones((m_tr, 1)), X_train], axis=1)

m_te = X_test.shape[0]
X_test_b = np.concatenate([np.ones((m_te, 1)), X_test], axis=1)

print(f'Shape X_train con bias: {X_train_b.shape}')
print(f'Shape X_test  con bias: {X_test_b.shape}')

### 4. Implementación del Modelo — Regresión Logística Multiclase (OvR)

#### 4.1 Función Sigmoidea

La hipótesis de la regresión logística binaria es:

$$h_\theta(x) = g(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

donde $g$ es la función sigmoidea. Para valores muy positivos entrega probabilidades cercanas a 1; para valores muy negativos, cercanas a 0.

In [ ]:
def sigmoid(z):
    """Función sigmoidea — opera sobre escalares, vectores y matrices."""
    z = np.array(z, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-z))

# Verificación rápida
print(f'sigmoid(0)   = {sigmoid(0)}   (esperado: 0.5)')
print(f'sigmoid(100) = {sigmoid(100)} (esperado: ~1.0)')
print(f'sigmoid(-100)= {sigmoid(-100)} (esperado: ~0.0)')

#### 4.2 Función de Costo y Gradiente

La función de costo de la regresión logística binaria es la **entropía cruzada binaria**:

$$J(\theta) = \frac{1}{m} \sum_{i=1}^{m} \left[ -y^{(i)} \log\left(h_\theta(x^{(i)})\right) - \left(1-y^{(i)}\right)\log\left(1 - h_\theta(x^{(i)})\right) \right]$$

El gradiente respecto a cada parámetro $\theta_j$ es:

$$\frac{\partial J}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^m \left(h_\theta(x^{(i)}) - y^{(i)}\right) x_j^{(i)}$$

En el esquema OvR se construye un vector binario $y_k$ para cada clase $k$: vale 1 si la etiqueta original es $k$, 0 en caso contrario.

In [ ]:
def costFunction(theta, X, y):
    """
    Calcula costo J y gradiente para regresión logística binaria.
    Devuelve (J, grad) — compatible con scipy.optimize.minimize.
    """
    m = y.size
    h = sigmoid(X.dot(theta))
    eps = 1e-12  # para estabilidad numérica en log
    J = (1/m) * np.sum(-y * np.log(h + eps) - (1 - y) * np.log(1 - h + eps))
    grad = (1/m) * X.T.dot(h - y)
    return J, grad

#### 4.3 Descenso por el Gradiente

Se implementa el algoritmo iterativo de descenso por el gradiente. En cada iteración se actualiza simultáneamente todos los parámetros $\theta$ en la dirección opuesta al gradiente, con tasa de aprendizaje $\alpha$:

$$\theta := \theta - \frac{\alpha}{m} X^T (h_\theta(X) - y)$$

Se registra el costo $J$ en cada iteración para poder graficar la curva de convergencia.

In [ ]:
def descensoGradiente(theta, X, y, alpha, num_iters):
    """
    Optimiza theta mediante descenso por el gradiente.
    Retorna (theta_opt, historial_de_costo).
    """
    m = y.shape[0]
    theta = theta.copy()
    J_history = []
    for _ in range(num_iters):
        h = sigmoid(X.dot(theta))
        theta -= (alpha / m) * X.T.dot(h - y)
        J, _ = costFunction(theta, X, y)
        J_history.append(J)
    return theta, J_history

### 5. Entrenamiento del Modelo OvR

Se entrena un clasificador binario por cada una de las 6 clases. Para la clase $k$, el vector de etiquetas de entrenamiento se transforma a binario: $y_k^{(i)} = 1$ si la etiqueta original es $k$, $0$ si no.

Al finalizar se almacenan los parámetros $\theta_k$ y el historial de costo de cada clasificador.

In [ ]:
NUM_CLASES = len(CLASES)
alpha     = 0.1
num_iters = 500
n_params  = X_train_b.shape[1]

all_theta     = np.zeros((NUM_CLASES, n_params))  # cada fila es theta_k
all_J_history = []

for k in range(NUM_CLASES):
    y_k = (y_train == k).astype(float)  # vector binario para clase k
    theta_k = np.zeros(n_params)
    theta_k, J_hist_k = descensoGradiente(theta_k, X_train_b, y_k, alpha, num_iters)
    all_theta[k]  = theta_k
    all_J_history.append(J_hist_k)
    print(f'Clase {k} ({nombre_clases[k]:12s}) — Costo final: {J_hist_k[-1]:.4f}')

print('\nEntrenamiento completado.')

### 6. Gráficas de Costo

La curva de costo debe descender monotónicamente y estabilizarse. Un descenso limpio indica que la tasa de aprendizaje $\alpha$ es adecuada y que el modelo está convergiendo hacia un mínimo.

In [ ]:
fig, axes = pyplot.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for k in range(NUM_CLASES):
    axes[k].plot(np.arange(num_iters), all_J_history[k], lw=2, color='steelblue')
    axes[k].set_title(f'Clase {k}: {nombre_clases[k]}')
    axes[k].set_xlabel('Iteraciones')
    axes[k].set_ylabel('Costo J')
    axes[k].grid(True, alpha=0.3)

pyplot.suptitle('Curvas de Convergencia del Costo — OvR ExtraSensory', fontsize=14, fontweight='bold')
pyplot.tight_layout()
pyplot.show()

### 7. Predicción y Evaluación

#### 7.1 Función de predicción OvR

Para predecir la clase de un nuevo ejemplo $x$, se evalúa cada clasificador $k$ y se obtiene la probabilidad $h_{\theta_k}(x)$. La clase predicha es aquella con la mayor probabilidad:

$$\hat{y} = \arg\max_k \ h_{\theta_k}(x)$$

In [ ]:
def predecir_ovr(all_theta, X):
    """
    Predicción OvR: devuelve la clase con mayor probabilidad para cada ejemplo.
    all_theta: (K x n+1), X: (m x n+1)
    """
    probs = sigmoid(X.dot(all_theta.T))  # (m x K)
    return np.argmax(probs, axis=1)

In [ ]:
# Precisión en entrenamiento
y_pred_train = predecir_ovr(all_theta, X_train_b)
acc_train = np.mean(y_pred_train == y_train) * 100
print(f'Precisión en ENTRENAMIENTO: {acc_train:.2f}%')

# Precisión en prueba (datos que el modelo nunca vio)
y_pred_test = predecir_ovr(all_theta, X_test_b)
acc_test = np.mean(y_pred_test == y_test) * 100
print(f'Precisión en PRUEBA       : {acc_test:.2f}%')

#### 7.2 Matriz de Confusión

La matriz de confusión muestra cuántos ejemplos de cada clase real fueron clasificados correctamente (diagonal) o confundidos con otra clase (fuera de la diagonal). Permite identificar qué actividades son más difíciles de distinguir.

In [ ]:
def confusion_matrix_manual(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm

cm = confusion_matrix_manual(y_test, y_pred_test, NUM_CLASES)

fig, ax = pyplot.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
pyplot.colorbar(im, ax=ax)

tick_marks = np.arange(NUM_CLASES)
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(nombre_clases, rotation=45, ha='right')
ax.set_yticklabels(nombre_clases)

thresh = cm.max() / 2.0
for i in range(NUM_CLASES):
    for j in range(NUM_CLASES):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > thresh else 'black')

ax.set_ylabel('Clase Real')
ax.set_xlabel('Clase Predicha')
ax.set_title('Matriz de Confusión — Conjunto de Prueba')
pyplot.tight_layout()
pyplot.show()

#### 7.3 Precisión por Clase

Se calcula la precisión individual de cada clase para identificar cuáles están bien aprendidas y cuáles requieren más datos o ajuste de hiperparámetros.

In [ ]:
precisiones_por_clase = []
for k in range(NUM_CLASES):
    mask_k = y_test == k
    if mask_k.sum() > 0:
        acc_k = np.mean(y_pred_test[mask_k] == k) * 100
    else:
        acc_k = 0.0
    precisiones_por_clase.append(acc_k)
    print(f'{nombre_clases[k]:12s}: {acc_k:.2f}%')

pyplot.figure(figsize=(9, 4))
bars = pyplot.bar(nombre_clases, precisiones_por_clase, color='teal', edgecolor='black')
pyplot.axhline(acc_test, color='red', linestyle='--', label=f'Precisión global: {acc_test:.2f}%')
pyplot.xlabel('Clase / Actividad')
pyplot.ylabel('Precisión (%)')
pyplot.title('Precisión por Clase — Conjunto de Prueba')
pyplot.ylim(0, 110)
pyplot.legend()
for bar, val in zip(bars, precisiones_por_clase):
    pyplot.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
               f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
pyplot.tight_layout()
pyplot.show()

### 8. Ejemplos de Predicción Individual

Se toman 10 ejemplos del conjunto de prueba y se compara la clase predicha con la clase real para verificar visualmente el funcionamiento del modelo.

In [ ]:
print('Predicciones en ejemplos del conjunto de prueba:')
print(f'{"Ejemplo":>8}  {"Real":>14}  {"Predicho":>14}  {"Correcto":>8}')
print('-' * 55)

np.random.seed(7)
indices = np.random.choice(len(y_test), 10, replace=False)

for i in indices:
    real  = nombre_clases[y_test[i]]
    pred  = nombre_clases[y_pred_test[i]]
    ok    = '✓' if y_test[i] == y_pred_test[i] else '✗'
    print(f'{i:>8}  {real:>14}  {pred:>14}  {ok:>8}')

### 9. Conclusiones

- El dataset ExtraSensory cumple los requisitos: **n = 225 features** y **m > 50 000 ejemplos**.
- Se aplicó preprocesamiento completo con Pandas: imputación de NaN, normalización Z-score y balanceo por submuestreo.
- El esquema **One-vs-Rest** permitió extender la regresión logística binaria a un problema de 6 clases sin modificar el algoritmo central.
- Las curvas de costo muestran convergencia para todos los clasificadores, validando la elección de $\alpha = 0.1$ y 500 iteraciones.
- La precisión global en el conjunto de prueba (20% de datos nunca vistos durante el entrenamiento) refleja la capacidad de generalización del modelo.
- La matriz de confusión permite identificar qué pares de actividades son más confundibles (por ejemplo, STANDING vs SITTING), lo cual orienta mejoras futuras.

**Repositorio GitHub:** https://github.com/jogaesmo/Escobar_Moscoso_Jorge_Gabriel_IA_1